In [1]:
import duckdb
from functions.evaluation import evaluate
from networks.cnn_network_large_kernels import CNNModel
from networks.cnn_network_large_kernels import build_dataloaders


con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index

train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
CNN.reset_weights()
CNN.retrain(cnn_train_dl,cnn_val_dl,patience=15)

Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 209,186
  -> ny bästa modell sparad till ../models/convolution_model_large_kernels.pth
Epoch   0 | train: 1.7565 | val: 0.6671 | acc: 82.96% | AUC: 0.806  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_large_kernels.pth
Epoch   1 | train: 0.5891 | val: 0.4621 | acc: 90.23% | AUC: 0.857  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_large_kernels.pth
Epoch   2 | train: 0.4717 | val: 0.3748 | acc: 90.51% | AUC: 0.881  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_large_kernels.pth
Epoch   3 | train: 0.4426 | val: 0.4540 | acc: 79.62% | AUC: 0.892  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_large_kernels.pth
Epoch   4 | train: 0.4314 | val: 0.3292 | acc: 93.64% | AUC: 0.896  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_large_kernels.pth
Epoch   5 

In [2]:
con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


CNN = CNNModel()
CNN.retrain_with_k_fold(k_fold_rows)

Totalt antal utan m-komponent i K-fold poolen: 18848
Totalt antal med m-komponent i K-fold poolen: 2692
Total parameters: 209,186
--- Startar 10-Fold Cross Validation ---

 FOLD 1/10
  -> ny bästa modell sparad till ../models/convolution_model_large_kernels_fold1.pth
Epoch   0 | train: 1.1920 | val: 0.6623 | acc: 86.58% | AUC: 0.725  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_large_kernels_fold1.pth
Epoch   1 | train: 0.5792 | val: 0.5172 | acc: 87.37% | AUC: 0.825  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_large_kernels_fold1.pth
Epoch   2 | train: 0.4561 | val: 0.4451 | acc: 87.41% | AUC: 0.859  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_large_kernels_fold1.pth
Epoch   3 | train: 0.4272 | val: 0.4368 | acc: 90.25% | AUC: 0.868  | LR: 0.001
Epoch   4 | train: 0.4210 | val: 0.4310 | acc: 83.74% | AUC: 0.867  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_large_kernels_fo

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.082535,0.150277,95.448212,0.988574,0.955921,0.944444,1800,83,15,255
1,2,0.080284,0.140808,96.003717,0.984892,0.965462,0.922222,1817,65,21,249
2,3,0.142621,0.183484,97.352531,0.982494,0.987792,0.873606,1861,23,34,235
3,4,0.085772,0.163627,96.934510,0.985736,0.975584,0.925651,1838,46,20,249
4,5,0.090071,0.163949,95.260223,0.985231,0.956452,0.925651,1801,82,20,249
5,6,0.082995,0.113719,96.053853,0.989189,0.961804,0.951673,1813,72,13,256
6,7,0.250751,0.303182,88.068709,0.940175,0.882228,0.869888,1663,222,35,234
7,8,0.079793,0.185071,96.096654,0.982840,0.968667,0.907063,1824,59,25,244
8,9,0.078719,0.259956,96.053853,0.970219,0.971883,0.881041,1832,53,32,237
9,10,0.231988,0.295382,94.281729,0.938217,0.962806,0.802974,1812,70,53,216


In [3]:
test_rows = test_rows[test_rows['label'].isin([0,1])]
result = CNN.predict(test_rows)
_ = evaluate(result)

KeyError: 'prediction'